# v72 — GONG SimCLR 事前学習 → verifier init 交換実験

v70 (RESEARCH_LOG §2.22) で深層 verifier は fold2 AUC 0.638 vs conf 0.797 で棄却。
唯一残った仮説「小データで視覚特徴が学べない」を、無ラベル GONG 全日面コーパスでの
ドメイン事前学習で検証する。test 観測 ±6h のフレームは取得時に除外済み
(manifest に URL/sha256 記録)。

**棄却ゲート**: ssl init の fold2 AUC が imagenet init を上回らなければ棄却。


In [ ]:
import os, shutil, subprocess, sys, json
from pathlib import Path

IN = Path('/kaggle/input')
print(sorted(str(p.relative_to(IN)) for p in IN.rglob('*') if p.is_dir()))
# マウントレイアウト差に備えて glob で解決 (/kaggle/input/datasets/<user>/<slug> 型を含む)
corpus_dir = next(p.parent for p in IN.rglob('batch_*.npz'))
ver_dir = next(p.parent for p in IN.rglob('crops_fold0.npy'))
code_dir = next(p.parent for p in IN.rglob('pretrain_gong_ssl.py'))
WORK = Path('/kaggle/working')
print('corpus:', corpus_dir, sorted(p.name for p in corpus_dir.iterdir())[:4])
print('verifier:', ver_dir, sorted(p.name for p in ver_dir.iterdir()))
print('code:', code_dir, sorted(p.name for p in code_dir.iterdir()))
print('cuda:', __import__('torch').cuda.is_available(),
      __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else '')


In [ ]:
%%time
# SimCLR 事前学習 (ResNet18, 128px crop, NT-Xent)
r = subprocess.run([sys.executable, str(code_dir / 'pretrain_gong_ssl.py'),
                    '--corpus', str(corpus_dir),
                    '--epochs', '40', '--batch', '128',
                    '--out', str(WORK / 'resnet18_ssl.pt')],
                   capture_output=True, text=True)
print(r.stdout[-3000:]); print('STDERR:', r.stderr[-2000:])
assert r.returncode == 0


In [ ]:
%%time
# verifier 学習: ssl init (評価は fold1 select → fold2 frozen)
shutil.copytree(ver_dir, WORK / 'verifier_v1', dirs_exist_ok=True)
r = subprocess.run([sys.executable, str(code_dir / 'train_verifier.py'),
                    '--data', str(WORK / 'verifier_v1'),
                    '--epochs', '8', '--init', f'ssl:{WORK}/resnet18_ssl.pt',
                    '--out', str(WORK / 'verifier_ssl')],
                   capture_output=True, text=True)
print(r.stdout[-3000:]); print('STDERR:', r.stderr[-2000:])
assert r.returncode == 0


In [ ]:
%%time
# 同環境の imagenet init 対照 (v70 値との一致も確認する)
r = subprocess.run([sys.executable, str(code_dir / 'train_verifier.py'),
                    '--data', str(WORK / 'verifier_v1'),
                    '--epochs', '8', '--init', 'imagenet',
                    '--out', str(WORK / 'verifier_imagenet')],
                   capture_output=True, text=True)
print(r.stdout[-3000:]); print('STDERR:', r.stderr[-2000:])
assert r.returncode == 0


In [ ]:
# 結果サマリ — 棄却ゲート判定
ssl = json.load(open(WORK / 'verifier_ssl' / 'train_result.json'))
img = json.load(open(WORK / 'verifier_imagenet' / 'train_result.json'))
summary = {'ssl_fold2_auc': ssl.get('auc_fold2_verifier'),
           'imagenet_fold2_auc': img.get('auc_fold2_verifier'),
           'ssl_blend': ssl.get('auc_fold2_blend'),
           'imagenet_blend': img.get('auc_fold2_blend'),
           'conf_auc_fold2': ssl.get('auc_fold2_conf')}
print(json.dumps(summary, indent=2))
Path(WORK / 'v72_summary.json').write_text(json.dumps(
    {'ssl': ssl, 'imagenet': img, 'summary': summary}, indent=2))
